# Live-Coupled SWAN + AHP-TOPSIS Site Selection

---

## Overview

This notebook implements a **fully coupled** workflow that connects the SWAN spectral wave model to the AHP-TOPSIS multi-criteria decision-making framework. Unlike the standalone AHP-TOPSIS notebook (which uses tabular data), here **all criteria are derived directly from SWAN model output**, providing physically resolved nearshore wave fields that account for refraction, shoaling, depth-induced breaking, and wind forcing.

The notebook is **region-agnostic** — set the environment variable ``SWAN_REGION`` to ``california_bay`` or ``oman`` (or change the ``REGION`` variable in the first code cell below) and the whole pipeline reconfigures automatically via ``code/regions.py``.


In [1]:
import json
import os
import sys
import numpy as np

%matplotlib inline

## 1. Import SWAN Runner Utilities

We import the `swan_runner` module developed in the Model Setup notebook. This provides functions to modify SWAN input files, execute the model, and parse output.

In [ ]:
# ---------------------------------------------------------------------
# REGION SELECTION — pick the SWAN model to drive this notebook
# ---------------------------------------------------------------------
# Set REGION to one of: california_bay, oman.  Everything downstream is
# parameterised through ``code/regions.py``.
import os, sys, json
import numpy as np

# --- bootstrap: locate the repository and make code/ importable ---
import os, sys
REPO_ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(REPO_ROOT, 'code')):
    if os.path.dirname(REPO_ROOT) == REPO_ROOT:
        raise FileNotFoundError('Repository root not found (no code/ directory). '
                                'Start Jupyter from inside the repository.')
    REPO_ROOT = os.path.dirname(REPO_ROOT)
CODE_DIR = os.path.join(REPO_ROOT, 'code')
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
# --- end bootstrap ---

from regions import get_region, list_regions, summarize
import swan_runner

REGION = os.environ.get('SWAN_REGION', 'california_bay')
cfg = get_region(REGION)
print(summarize(REGION))

missing = cfg.check_inputs()
if missing:
    print('WARNING — some required inputs are missing on disk:')
    for _m in missing:
        print('  -', _m)

# Per-region figures directory for this notebook
FIGS_DIR = cfg.figs_dir('03_site_selection')
print(f'Figures will be saved to: {FIGS_DIR}')


## 2. Configuration

Define paths to the SWAN model directory and configure the candidate site grid. Sites are generated on a regular lat/lon grid covering the computational domain.

In [ ]:
# ----- Paths derived from region config -----
MODEL_DIR      = cfg.model_dir
INPUT_TEMPLATE = cfg.swan_input_path
RUN_INPUT      = os.path.join(MODEL_DIR, 'ahp_run')
OUTPUT_TABLE   = 'site_output.tab'
POINTS_FILE    = 'candidate_sites.dat'
OUTPUT_JSON    = os.path.join(FIGS_DIR, f'best_site_{cfg.key}.json')
OUTPUT_PLOT    = os.path.join(FIGS_DIR, f'ahp_topsis_map_{cfg.key}.png')

# ----- Candidate-site sampling grid (from region config) -----
LON_MIN, LON_MAX, LON_STEP = cfg.site_grid_lon
LAT_MIN, LAT_MAX, LAT_STEP = cfg.site_grid_lat
DOMAIN_LON = (LON_MIN, LON_MAX)
DOMAIN_LAT = (LAT_MIN, LAT_MAX)

# ----- SWAN execution -----
SWAN_CMD    = 'swanrun'
SWAN_NPROC  = 8
SWAN_VARS   = ['HSIGN', 'RTP', 'DIR', 'DEPTH']
TIME_START  = cfg.t_start.strftime('%Y%m%d.%H%M%S')
TIME_STEP   = '1 HR'

# Sanity check: make sure the template exists before we try to copy it
if not os.path.exists(INPUT_TEMPLATE):
    raise FileNotFoundError(
        f'SWAN input template not found: {INPUT_TEMPLATE}. '
        f'Check cfg.swan_input_name = {cfg.swan_input_name!r}.'
    )

print(f"Region         : {cfg.display_name}  [{cfg.key}]")
print(f"Model directory: {MODEL_DIR}")
print(f"SWAN input     : {cfg.swan_input_name}")
print(f"Domain         : lat [{LAT_MIN:.2f}, {LAT_MAX:.2f}]  "
      f"lon [{LON_MIN:.2f}, {LON_MAX:.2f}]  step={LON_STEP:.3f}°")
print(f"Time origin    : {TIME_START}")


## 3. Generate Candidate Sites

Create a regular grid of candidate WEC deployment sites within the SWAN computational domain. Points of the sampling box that fall outside the (possibly rotated) SWAN grid are discarded.


In [ ]:
def generate_candidate_sites():
    """
    Regular grid of candidate sites within the region's sampling box, keeping
    only points that fall inside the (possibly rotated) SWAN computational
    grid. Points outside the grid come back from SWAN as exception values and
    would otherwise be mistaken for land/shore points.
    """
    lons = np.arange(DOMAIN_LON[0], DOMAIN_LON[1] + 1e-9, LON_STEP)
    lats = np.arange(DOMAIN_LAT[0], DOMAIN_LAT[1] + 1e-9, LAT_STEP)
    sites = [(float(lo), float(la)) for la in lats for lo in lons]
    inside = cfg.grid_contains([s[0] for s in sites], [s[1] for s in sites])
    n_out = int((~inside).sum())
    if n_out:
        print(f"Dropped {n_out} of {len(sites)} sampling points that lie "
              f"outside the SWAN grid.")
    return [s for s, ok in zip(sites, inside) if ok]

locations = generate_candidate_sites()
print(f"Generated {len(locations)} candidate sites for {cfg.display_name}")


## 4. Helper Functions: Criteria Computation

### Haversine Distance

Used to compute the great-circle distance between a candidate site and the nearest shoreline point.

### Circular Standard Deviation

Wave direction is a circular variable (0-360 deg). We use the circular standard deviation $\sigma_c = \sqrt{-2 \ln R}$ where $R$ is the mean resultant length. Low directional variability is preferred for WEC alignment.

In [5]:
def haversine(lat1, lon1, lat2, lon2):
    """Haversine distance in km (scalar)."""
    R = 6371.0
    la1, lo1 = np.radians(lat1), np.radians(lon1)
    la2, lo2 = np.radians(lat2), np.radians(lon2)
    a = (np.sin((la2 - la1) / 2) ** 2
         + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2)
    return 2 * R * np.arcsin(np.sqrt(a))


def circular_std(angles_deg):
    """Circular standard deviation for directional data."""
    rad = np.radians(angles_deg)
    R = np.sqrt(np.mean(np.cos(rad)) ** 2 + np.mean(np.sin(rad)) ** 2)
    return float(np.degrees(np.sqrt(-2 * np.log(max(R, 1e-10)))))

### Compute All 5 Criteria from SWAN TABLE Output

The SWAN TABLE contains one row per (timestep x location). We reshape into `(n_times, n_sites)` and compute temporal statistics for each site. Sites on land (depth <= 0 or Hs < 0) are excluded.

In [ ]:
def compute_criteria_from_swan(table_data, locations):
    """
    Compute 5 criteria for each candidate site from SWAN TABLE output.

    Two passes: first every land/shore point is identified, then distance to
    shore is computed against *all* of them. (A single pass would only see
    shore points that happen to come earlier in the loop, making the result
    depend on site order.)
    """
    n_sites = len(locations)
    n_total = len(table_data['Hsig'])
    n_times = n_total // n_sites

    if n_times * n_sites != n_total:
        raise ValueError(
            f"TABLE rows ({n_total}) not divisible by sites ({n_sites}).")

    Hs = table_data['Hsig'].reshape(n_times, n_sites)
    Tp = table_data['RTpeak'].reshape(n_times, n_sites)
    Dir = table_data['Dir'].reshape(n_times, n_sites)
    Dep = table_data['Depth'].reshape(n_times, n_sites)

    rho_g2 = 1025 * 9.81 ** 2 / (64 * np.pi)

    # --- Pass 1: land / shore points (dry or SWAN exception values) -------
    is_shore = (Dep.mean(axis=0) <= 1.0) | (Hs.mean(axis=0) < 0)
    shore = np.array([locations[j] for j in np.where(is_shore)[0]], dtype=float)
    if len(shore) == 0:
        print("  WARNING: no land/shore points inside the candidate grid; "
              "distance-to-shore is set to NaN and will be dropped.")

    # --- Pass 2: criteria for wet sites ------------------------------------
    sites = []
    for j in np.where(~is_shore)[0]:
        hs, tp, dr, dep = Hs[:, j], Tp[:, j], Dir[:, j], Dep[:, j]

        valid = (hs > 0) & (tp > 0)
        if np.sum(valid) < 10:
            continue

        P = rho_g2 * hs[valid] ** 2 * tp[valid] / 1000
        mean_power = float(np.mean(P))
        consistency = 100.0 * np.sum(hs[valid] >= 1.0) / len(hs[valid])
        depth = float(np.mean(dep[dep > 0])) if np.any(dep > 0) else 0.0

        lon_j, lat_j = locations[j]
        if len(shore):
            dist_shore = float(np.min(haversine(lat_j, lon_j,
                                                shore[:, 1], shore[:, 0])))
        else:
            dist_shore = np.nan

        sites.append(dict(
            lon=lon_j, lat=lat_j,
            mean_power=round(mean_power, 2),
            consistency=round(consistency, 2),
            depth=round(depth, 1),
            dist_shore=round(dist_shore, 2),
            dir_variability=round(circular_std(dr[valid]), 2),
            mean_hs=round(float(np.mean(hs[valid])), 3),
            mean_tp=round(float(np.mean(tp[valid])), 3),
        ))

    return sites


## 5. AHP-TOPSIS Implementation

Self-contained AHP and TOPSIS functions (same algorithms as the standalone notebook, included here for completeness).

In [7]:
def ahp_weights(A):
    n = A.shape[0]
    evals, evecs = np.linalg.eig(A)
    idx = np.argmax(evals.real)
    w = np.abs(evecs[:, idx].real)
    w /= w.sum()
    ci = (evals[idx].real - n) / (n - 1) if n > 1 else 0.0
    ri = {1: 0, 2: 0, 3: .58, 4: .90, 5: 1.12}.get(n, 1.49)
    return w, float(ci / ri) if ri else 0.0


def topsis(X, w, benefit):
    R = X / np.sqrt((X ** 2).sum(axis=0))
    V = R * w
    nc = V.shape[1]
    pis = np.array([V[:, j].max() if benefit[j] else V[:, j].min() for j in range(nc)])
    nis = np.array([V[:, j].min() if benefit[j] else V[:, j].max() for j in range(nc)])
    sp = np.sqrt(((V - pis) ** 2).sum(axis=1))
    sm = np.sqrt(((V - nis) ** 2).sum(axis=1))
    C = sm / (sp + sm)
    return C, np.argsort(C)[::-1]

## 6. Run the Full Pipeline

The cell below executes the complete workflow:

1. Check SWAN availability
2. Write candidate sites file
3. Prepare modified SWAN input
4. Run SWAN
5. Read output and compute criteria
6. AHP-TOPSIS ranking
7. Save and plot results

**Note:** This cell requires SWAN to be installed. If SWAN is not available, it will exit with an error message.

In [ ]:
SWAN_RAN = False   # set True only when this run completes; the plot cell checks it
print("=" * 65)
print(f"Live-Coupled SWAN + AHP-TOPSIS  --  {cfg.display_name}")
print("=" * 65)

# --- 0. Check SWAN ---
if not swan_runner.check_swan_available(SWAN_CMD):
    print(f"\n  WARNING: '{SWAN_CMD}' not found on PATH.")
    print("  Install SWAN and ensure the executable is accessible.")
    print("  See the repository README for installation instructions.")
    print("\n  Skipping SWAN execution. Using pre-computed results if available.")
else:
    # --- 1. Candidate sites ---
    print(f"\n[1/5] {len(locations)} candidate sites")

    # --- 2. Prepare SWAN input ---
    print("[2/5] Preparing SWAN input...")
    pts_path = os.path.join(MODEL_DIR, POINTS_FILE)
    swan_runner.write_points_file(pts_path, locations)

    var_str = ' '.join(SWAN_VARS)
    extra_cmds = [
        f"POINTS 'sites' FILE '{POINTS_FILE}'",
        f"TABLE 'sites' HEADER '{OUTPUT_TABLE}' {var_str} "
        f"OUTPUT {TIME_START} {TIME_STEP}",
    ]
    swan_runner.prepare_input(INPUT_TEMPLATE, RUN_INPUT + '.swn', extra_cmds)

    # --- 3. Run SWAN ---
    print("[3/5] Running SWAN (this may take several minutes)...")
    swan_runner.run_swan(MODEL_DIR, input_name='ahp_run',
                         swan_cmd=SWAN_CMD, nproc=SWAN_NPROC)

    # --- 4. Read output & compute criteria ---
    print("[4/5] Reading SWAN output & computing criteria...")
    table_path = os.path.join(MODEL_DIR, OUTPUT_TABLE)
    table_data = swan_runner.read_table(table_path)
    sites = compute_criteria_from_swan(table_data, locations)
    print(f"       --> {len(sites)} valid ocean sites")

    if len(sites) == 0:
        raise RuntimeError(
            "No valid ocean sites were produced. Check that the candidate grid "
            "actually overlaps the SWAN computational domain and that the run "
            "completed successfully (see norm_end / .prt)."
        )

    # --- 5. AHP-TOPSIS ---
    print("[5/5] AHP-TOPSIS ranking...")
    A = np.array([
        [1, 3, 5, 7, 9], [1/3, 1, 3, 5, 7], [1/5, 1/3, 1, 3, 5],
        [1/7, 1/5, 1/3, 1, 3], [1/9, 1/7, 1/5, 1/3, 1],
    ])
    crit = ["Power", "Consist", "Depth", "Shore", "DirVar"]
    benefit = [True, True, False, False, False]
    weights, cr = ahp_weights(A)
    print(f"       Weights: {dict(zip(crit, [f'{w:.3f}' for w in weights]))}")
    print(f"       CR = {cr:.4f} {'(OK)' if cr < 0.10 else '(REVISE!)'}")

    X = np.array([[s['mean_power'], s['consistency'], s['depth'],
                    s['dist_shore'], s['dir_variability']] for s in sites])
    if np.isnan(X[:, 3]).all():          # no shore points in the grid
        keep = [0, 1, 2, 4]
        X, benefit = X[:, keep], [benefit[i] for i in keep]
        weights = weights[keep] / weights[keep].sum()
        print("       Distance-to-shore unavailable -> criterion dropped, "
              "weights renormalised.")
    C, ranking = topsis(X, weights, benefit)

    # Print top 10
    n_show = min(10, len(ranking))
    print(f"\n{'Rank':<5}{'Lat':>7}{'Lon':>9}{'P':>8}{'C%':>6}{'D':>7}"
          f"{'Shr':>6}{'DV':>6}{'C*':>8}")
    print("-" * 55)
    for r in range(n_show):
        i = ranking[r]
        s = sites[i]
        print(f"{r+1:<5}{s['lat']:7.3f}{s['lon']:9.3f}{s['mean_power']:8.1f}"
              f"{s['consistency']:6.1f}{s['depth']:7.1f}"
              f"{s['dist_shore']:6.1f}{s['dir_variability']:6.1f}{C[i]:8.4f}")

    best = sites[ranking[0]]
    print(f"\n  Best site: ({best['lat']:.3f}N, {best['lon']:.3f}E) "
          f" C*={C[ranking[0]]:.4f}")

    # Save for GA script (tagged with region key)
    out = {'region': cfg.key, **best, 'c_star': round(float(C[ranking[0]]), 4)}
    with open(OUTPUT_JSON, 'w') as f:
        json.dump(out, f, indent=2)
    print(f"  Saved to {os.path.basename(OUTPUT_JSON)}")
    SWAN_RAN = True


## 7. Visualization: Site Ranking Map

Plot all candidate sites colored by their TOPSIS $C^*$ score. The best site is marked with a blue star.

In [ ]:
if not SWAN_RAN:
    print("(SWAN was skipped — no ranking to visualise.)")
else:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(9, 7))
    sc = ax.scatter([s['lon'] for s in sites], [s['lat'] for s in sites],
                    c=[C[i] for i in range(len(sites))],
                    cmap='RdYlGn', s=100, edgecolors='k', linewidths=.4)
    fig.colorbar(sc, ax=ax, label='C*', shrink=0.8)
    ax.plot(best['lon'], best['lat'], '*', color='blue', ms=16, mec='k',
            label=f"Best site  (C*={C[ranking[0]]:.3f})")
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(f'AHP-TOPSIS site ranking --  {cfg.display_name}')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_PLOT, dpi=140)
    plt.show()
    print(f"Saved map to {os.path.basename(OUTPUT_PLOT)}")


## Summary

This notebook demonstrated the complete SWAN-coupled AHP-TOPSIS site selection pipeline:

1. **SWAN resolves** the nearshore wave field with full physics (refraction, shoaling, breaking, etc.)
2. **Criteria are computed** directly from the resolved fields — no assumptions about offshore-to-nearshore transformation
3. **AHP weights** encode expert priorities (wave power >> logistics >> environment)
4. **TOPSIS ranking** provides an objective, distance-based ranking of all candidates
5. The **best site** is saved to `best_site.json` and passed to the GA array optimization notebook

### Next Steps

Proceed to `04_optimization/swan_coupled_ga_array.ipynb` to optimize the WEC array layout at the selected site.